# C1.1

## LLM runtime concepts

### How LLMs Generated output

To understand how LLMs work, let's create one. Say you've got an enormous corpus of text, a few books. Here's how you create an LLM from it.

#### 1. Tokenization

You first break down the dataset into sequence of words, and label each word as a unique number. The key value pairs may look like this:

1 - a
2 - an
3 - apple
...

So in your dataset, if you have the sentence:
Ram ate an apple.

That sentence in terms of numbers is:
889 17 2 3 (assuming that 889 corresponds to Ram and 17 corresponds to ate in the above key value pairs.

#### 2. Next token prediction

Suppose you can train a model like this:

You give the model an enormous dataset of numbers, like 31, 39, 45, 68, 31, 39, 32, 14, 22, ...

You train the model on this data. You can then ask it to predict the next number based on the sequence of numbers before. Like what comes after 31, 39. The model gives output that it's 80% probability that the next number is 45, 10% probability that it's 32, 0% that it's 31, etc.


This is exactly how LLMs are trained. Use the corpus of text labelled as numbers to train the number predictor. Then convert the predicted number back to word from the key value storage as your prediction for the next word.


That's it! That's the entire workflow of LLMs. Of course, there are some bells and whistles that make up the entire workflow, but this is the core engine of the LLM. The bells are whistles are as follows:

1. You know how LLMs write paragraphs, not just predict the next word. That's achived by running the prediction function again and again. Say your first line was "He jumped" and the word-predictor predicted "over", then the entire sentence "He jumped over" is passed back to the word-predictor, which predicts (say) "the", and the new sentence becomes "He jumped over the". This process continues till a special _stop_ token is outputted from the next word predictor.
2. Stop keyword: As you saw above, the word-predictor is called again and again, but for how long? Till a specific special stop token is predicted. This is a special word that we added to the dataset at strategic places(end of chapter, book, etc). So the model learns to predict this word at some point. When our code sees that word as the output, we break the loop. For most models, this word is <|end|>.
3. Instruction training: You think that LLMs don't really autocomplete your input, they answer questions. Well, behind the scenes, they still autocomplete. It's just that they are trained over an enormous amount of text blocks in the format:

```
This is a conversation between a human and an AI assistant.

Human: What's the size of the Earth?

AI assistant: It's somewhere between large and very large.
```
Training on such examples is called Instruction training. That's why on Hugging face, you'll see the label Instruct in front of model. It means that it's trained on instruction examples.

### What are tokens

You saw how we converted each word into a specific number. In practice, it's not words that we convert to tokens. Space is it's own token. Often times, space with word is a separate token, like " red" is a different token than "red" or "Red". Periods, punchuation marks are their own tokens. Some words are combination of multiple tokens, like "counterexample" may consist of two tokens: counter and example.

So these tokens, converted to numbers are used for predicting the next word. For us, the number of tokens sent/received is very important. 

If you asked the LLM, "How much is 7 times 4?", after tokenization, it may look like "43, 21, 97, 21, 63, 21, 7, 95, 21, 4, 88". If the model generates "It's 28", its sequence of output tokens may look like "54, 77, 21, 28". Thus, you sent 11 tokens and got back 4 tokens. Mostly, almost everywhere, generating the next output token is the most expensive step in the pipeline, thus more tokens generated - high cost. Hence, you see much higher price/token in output than input(generally measured in $/1 M tokens).

Here's an example of converting words to tokens

In [5]:
# nltk - natural language toolkit
import nltk
from nltk.tokenize import word_tokenize

# punkt is a tokenizer model
nltk.download('punkt_tab')

text = "Natural language toolkit is the natural choice for Natural Language processing."
tokens = word_tokenize(text)
print("Tokens: ", tokens)

Tokens:  ['Natural', 'language', 'toolkit', 'is', 'the', 'natural', 'choice', 'for', 'Natural', 'Language', 'processing', '.']


[nltk_data] Downloading package punkt_tab to /Users/rick/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


### What are Context Windows?

Previously, we said that given a seqeunce of tokens, LLM generates the next token. THe maximum number of previous tokens that we can supply is called its context length. 

When we say that an LLM has context or memory about your past chats, what's really happening is that the context/memory is passed as text to the word predictor as context. Like the prompt becomes:

```
This is a conversation between an AI assistant and a person.
Context: {context/memory}

User: {query}
AI assistant: ...{starts predicting next token}
```

Few important notes about context window:
1. If you pass text with more tokens than the max-context length for a model(generally called just context-length), it'll throw an error. You may have seen these errors in LLM APIs(we'll see them later as well).
2. This is the reason why context-engineering is important, you need to pack as much information you can in the least amount of tokens possible.
3. Thesedays, the context length of large language floats around 100,000 for small language models, and 1,000,000 for very large language models.
4. Just because the context length for a model is very large and input tokens don't cost much doesn't mean you should always use a lot of input tokens. It has been observed that providing a lot of irrelevant information in the prompt creates a needle-in-haystack problem, i.e., the models performance drops with more context to take care of.
5. For effective, context management, techniques like RAG, RLM, etc are used.

### Temperature

Previously, we saw how the model outputs probabilities for each possible next token. 80% for token 43, 10% for token 11, etc. Temperature is a parameter that we pass to the model while predicting the next token which has the effect that, the tokens having less probability of being the next token get a slight boost in their probability and the ones with higher than others get reduction in probability, i.e., the probability distribution becomes slightly more uniform.

Higher the value of the temperature parameter, higher is the boost gained by the less probable tokens.


The effect of higher temperature is that the model appears to have a more random behaviour in predicting the next token. Sometimes it results in more creative answers, sometimes it also gives something that's wild and may be considered as stupid.

Ideally, if you want predictable behavior, tone and answers from the LLM, set temperature to a low value, and if you want very creative answers, set temperature to high value. For many models, default value of temperature is 0.7 if not passed to it. Also, model providers cap the value of temperature(to 2 most of the times) so that the model doesn't produce gibberish response.

**Please note that Anthropic sdk has deprecated the use of temperature parameter in their APIs, so it's not possible to change the temperature of Anthropic models.**

### Model Families

For this course, we'll look into 3 main model families: OpenAI, Google, Anthropic.
Their flagship LLM families are called ChatGPT, Gemini and Claude respectively.

To use these models, you'll need an APi key for authorization. To make LLM calls, you can use either http requests or their sdks. We'll show you how to do that in further sections.

#### Google

As of writing this, Google has 2 families of models:
Gemini
Gemma

Gemini - The gemini models are very large, closed source models served on cloud. There are 4 layers of Gemini models - Gemini Flash Lite, Gemini Flash, Gemini Pro and Gemini Thinking.

Gemini Pro is much bigger model than Gemini Flash, which itself if much bigger than Gemini Flash Lite. As a rule of thumb, larger models have more information about the world, i.e., more data, but larget models take longer for inference. 

Gemini Flash and Gemini Flash Lite don't have reasoning built into them. Gemini Pro and Gemini Thinking do have reasoning built into them. 
<!-- We haven't explained what reasoning is, look into that.-->

To use them, you need API keys. You can either use Google's subscription model or you can use pay-as-you-go model. You can get an API key for free and use it for inference, but it'll have rate limits on requests/minute, (output)tokens/minute, requests/day and tokens/day. You can use their AI pro subscription for higher rate limits and AI ultra subscription for even higher limits.

In the pay-as-you-go model, you pay according to your inference. The cost for each input is measured according to the model's cost per token, typically stated as $/1M tokens. This cost is different for input tokens and output tokens. Total cost for one inference is the sum of input token costs and output token costs. Typically, output token costs are ~100x more expensie compared to input token costs.

Gemini models are very useful for their very large context windows, surpassing 1 Million tokens for Gemini Pro. This means that they can hold entire books as part of their contexts.